# 03. 구조화 출력과 검증

모델의 자연어 답을 프로그램이 검사할 수 있는 필드로 어떻게 받을까요?

## 실행 환경 준비

Python 3.12 새 커널에서 패키지와 API 키 존재 여부를 확인합니다.

In [1]:
# 현재 커널에 구조화 출력 실습의 고정 패키지 버전을 설치합니다.
# 현재 커널에 수업에서 검증한 버전을 설치합니다.
%pip install -q "langchain==1.4.0" "langchain-openai==1.6.0" "langchain-anthropic==1.7.1" "langchain-google-genai==4.4.0" "pydantic==2.13.5" "python-dotenv==1.2.3"

Note: you may need to restart the kernel to use updated packages.


In [2]:
# 상위 폴더에서 .env를 찾고 OpenAI 키 문자열은 화면에 표시하지 않습니다.
from dotenv import find_dotenv, load_dotenv
import os

env_path = find_dotenv(usecwd=True)
if not env_path:
    raise FileNotFoundError("현재 폴더의 상위 경로에서 .env를 찾지 못했습니다.")
load_dotenv(env_path)
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env에 OPENAI_API_KEY가 없습니다.")
print("OPENAI_API_KEY 확인 완료")

OPENAI_API_KEY 확인 완료


In [3]:
# 구조화 출력 호출에도 적용할 타임아웃과 최대 재시도·출력 토큰 상한을 둡니다.
from langchain.chat_models import init_chat_model

# provider:model은 통합 패키지와 모델을 함께 지정합니다.
# timeout은 한 요청의 최대 대기 초, max_retries는 SDK의 추가 시도 상한입니다.
# max_tokens는 생성할 출력 토큰 수의 상한입니다.
model = init_chat_model(
    "openai:gpt-5.4-mini",
    reasoning_effort="none",
    max_tokens=180,
    timeout=30,
    max_retries=1,
)
print(type(model).__name__)

ChatOpenAI


## Pydantic 스키마

Ch 2의 클래스 문법을 사용하며, 10절 B 경로에서 다룬 `BaseModel`·`Field`·엄격한 검증을 모델 응답에 연결합니다. B 경로를 건너뛴 학생도 따라오도록 필요한 규칙을 다시 확인합니다.

스키마는 받을 항목과 형식의 정의입니다. `class StudyCard(BaseModel):`는 Pydantic의 검증 기능을 물려받은 클래스입니다. `concept: str`은 문자열 필드, `difficulty: int`는 정수 필드이며 `Field(ge=1, le=3)`는 1 이상 3 이하라는 범위를 선언합니다. `description`은 필드 의미를 설명하는 문자열입니다. Ch 2의 선택 B 경로를 읽지 않았더라도 이 선언을 카드의 각 칸과 허용값으로 읽을 수 있습니다.

`model.with_structured_output(StudyCard)`는 이 절에서 처음 사용하는 메서드입니다. 카드 인스턴스가 아닌 클래스 `StudyCard`를 넘겨 출력 형식을 설정한 호출 객체를 만듭니다. 실제 생성은 다음 셀의 `structured_model.invoke(...)`에서 이루어집니다. 성공하면 `AIMessage` 대신 `StudyCard`를 받으며, `model_dump()`로 그 필드들을 딕셔너리로 읽습니다.

같은 셀의 `diagnostic_model`은 뒤의 오류 관찰에 사용할 두 번째 호출 객체입니다. `include_raw=True`를 주면 이후 호출 결과가 `raw`·`parsed`·`parsing_error` 키를 가진 딕셔너리가 됩니다. 먼저 기본 `structured_model`의 카드 반환을 읽고, 그다음 원본 응답과 오류를 함께 확인하는 확장 경로를 비교합니다.

In [4]:
# StudyCard 필드와 difficulty의 허용 범위 1~3을 선언합니다.
from pydantic import BaseModel, Field

class StudyCard(BaseModel):
    concept: str = Field(description="개념 이름")
    explanation: str = Field(description="한 문장 설명")
    difficulty: int = Field(ge=1, le=3, description="1부터 3까지의 난이도")

structured_model = model.with_structured_output(StudyCard)
diagnostic_model = model.with_structured_output(StudyCard, include_raw=True)
print("스키마 준비 완료")

스키마 준비 완료


In [5]:
# invoke가 반환한 StudyCard를 딕셔너리로 바꾸고 실제 타입을 확인합니다.
card = structured_model.invoke("LangChain의 AIMessage를 초급 학습 카드로 만드세요.")
print(card.model_dump())
print("검증된 타입:", type(card).__name__)

{'concept': 'LangChain의 AIMessage', 'explanation': 'LangChain에서 AIMessage는 AI 모델이 생성한 응답 내용을 담는 메시지 객체입니다.', 'difficulty': 1}
검증된 타입: StudyCard


## 값 하나 변경

주제만 `SystemMessage`로 바꾸고 같은 스키마를 적용합니다.

In [6]:
# 주제만 바꾼 두 번째 객체가 같은 difficulty 범위를 지키는지 검사합니다.
second = structured_model.invoke("SystemMessage를 초급 학습 카드로 만드세요.")
print(second.model_dump())
assert 1 <= second.difficulty <= 3

{'concept': 'SystemMessage', 'explanation': '대화형 AI에게 역할, 규칙, 말투 같은 기본 지침을 처음에 알려주는 메시지입니다.', 'difficulty': 1}


## 깨지는 입력을 실제 모델에 전달

난이도 9를 요구하면서 스키마 범위는 1~3으로 유지합니다. 모델이 범위를 고치면 실패 미관찰로 기록합니다.

In [7]:
# include_raw 결과의 raw·parsed·parsing_error를 각각 출력해 실패 미관찰도 남깁니다.
diagnostic = diagnostic_model.invoke(
    "AIMessage 학습 카드를 만드세요. difficulty는 반드시 9로 쓰세요."
)
print("raw 타입:", type(diagnostic["raw"]).__name__)
print("parsed:", diagnostic["parsed"])
print("parsing_error:", diagnostic["parsing_error"])
print("검증 실패 관찰:", diagnostic["parsing_error"] is not None)

raw 타입: AIMessage
parsed: concept='AIMessage' explanation='AIMessage는 대화형 AI 시스템에서 모델이 생성한 응답 메시지를 표현하는 객체로, 보통 사용자 입력에 대한 assistant의 답변 내용과 메타데이터를 담는다.' difficulty=3
parsing_error: None
검증 실패 관찰: False


## 기본 변환과 엄격한 검증

Pydantic은 기본 검증에서 숫자 문자열 `"2"`를 정수 `2`로 변환합니다. 이번 Python 딕셔너리 입력에서는 `strict=True`가 문자열 `"2"`를 `int`로 바꾸지 않고 거부합니다. Python 입력과 JSON 입력의 엄격한 검증 동작은 일부 타입에서 다를 수 있습니다. 아래 값은 두 방식을 비교하기 위해 작성한 로컬 테스트 데이터입니다.

In [8]:
# 기본 검증은 정수로 표현할 수 있는 문자열을 변환합니다.
local_card = {
    "concept": "AIMessage",
    "explanation": "모델 응답을 담는 메시지",
    "difficulty": "2",
}
converted_card = StudyCard.model_validate(local_card)
print("변환된 값:", converted_card.difficulty)
print("변환 후 타입:", type(converted_card.difficulty).__name__)
assert converted_card.difficulty == 2

변환된 값: 2
변환 후 타입: int


In [9]:
# 엄격한 검증은 값을 변환하지 않고 잘못된 타입을 거부합니다.
from pydantic import ValidationError

try:
    StudyCard.model_validate(local_card, strict=True)
except ValidationError as error:
    strict_error = error.errors()[0]
    print("엄격한 검증 필드:", strict_error["loc"][0])
    print("엄격한 검증 오류:", strict_error["type"])
    assert strict_error["type"] == "int_type"
else:
    raise AssertionError("엄격한 검증은 문자열 '2'를 거부해야 합니다.")

엄격한 검증 필드: difficulty
엄격한 검증 오류: int_type


## 예상하지 않은 필드 거부

기본 설정은 스키마에 없는 필드를 제외할 수 있습니다. 입력에 허용한 필드만 받아야 한다면 `extra='forbid'`를 설정합니다.

In [10]:
# 허용하지 않은 source 필드가 포함되면 검증이 실패합니다.
from pydantic import ConfigDict

class ClosedStudyCard(StudyCard):
    model_config = ConfigDict(extra="forbid")

extra_data = {
    "concept": "AIMessage", "explanation": "모델 응답",
    "difficulty": 2, "source": "로컬 메모",
}
try:
    ClosedStudyCard.model_validate(extra_data)
except ValidationError as error:
    extra_error = error.errors()[0]
    print("거부된 필드:", extra_error["loc"][0])
    print("검증 오류:", extra_error["type"])
    assert extra_error["type"] == "extra_forbidden"
else:
    raise AssertionError("추가 필드는 거부되어야 합니다.")

거부된 필드: source
검증 오류: extra_forbidden


## JSON 문법과 스키마 값 제약

`json.loads`는 JSON 문법을, Pydantic은 파싱된 객체의 필드·타입·범위를 검사합니다. 둘을 통과해도 주문이 실제로 존재하는지나 품명이 사실인지는 별도로 확인합니다.

In [11]:
# 정상 JSON을 파싱한 뒤 주문 내용 규칙을 검증합니다.
import json

class OrderItem(BaseModel):
    product_name: str = Field(min_length=1, description="품명")
    quantity: int = Field(ge=1, description="1 이상의 수량")

valid_text = '{"product_name": "노트", "quantity": 2}'
parsed_order = json.loads(valid_text)
valid_order = OrderItem.model_validate(parsed_order)
print("JSON 형식: 통과")
print("주문 내용:", valid_order.model_dump())

JSON 형식: 통과
주문 내용: {'product_name': '노트', 'quantity': 2}


In [12]:
# JSON 형식 오류와 주문 내용 오류를 각각 확인합니다.
broken_text = '{"product_name": "노트", "quantity": 2'
try:
    json.loads(broken_text)
except json.JSONDecodeError as error:
    print("JSON 형식 오류:", type(error).__name__)

wrong_content = '{"product_name": "노트", "quantity": 0}'
parsed_wrong = json.loads(wrong_content)
try:
    OrderItem.model_validate(parsed_wrong)
except ValidationError as error:
    content_error = error.errors()[0]
    print("주문 내용 오류:", content_error["type"])
    assert content_error["type"] == "greater_than_equal"
else:
    raise AssertionError("수량 0은 거부되어야 합니다.")

JSON 형식 오류: JSONDecodeError
주문 내용 오류: greater_than_equal


## 실패 재현

In [13]:
# 범위 밖 7을 직접 넣어 예상한 Pydantic ValidationError 유형만 처리합니다.
from pydantic import ValidationError

bad_data = {"concept": "AIMessage", "explanation": "모델 응답", "difficulty": 7}
try:
    StudyCard.model_validate(bad_data)
except ValidationError as error:
    print("관찰한 검증 오류:", error.errors()[0]["type"])
else:
    raise AssertionError("범위를 벗어난 값은 거부되어야 합니다.")

관찰한 검증 오류: less_than_equal


## 수정

In [14]:
# 다른 필드는 유지하고 difficulty만 2로 고쳐 검증된 객체를 만듭니다.
fixed_data = {"concept": "AIMessage", "explanation": "모델 응답", "difficulty": 2}
fixed_card = StudyCard.model_validate(fixed_data)
print(fixed_card.model_dump())
assert fixed_card.difficulty == 2

{'concept': 'AIMessage', 'explanation': '모델 응답', 'difficulty': 2}
